# Fase 15: Contoh citra dataset untuk naskah (Kaggle, CPU, tanpa training)

**Tujuan.** Memilih satu contoh citra per kelas (8 kelas steel-pipe, 6 kelas SWRD) untuk gambar dataset di
Section IV-A. Gambar hanya memuat citra dan kotak *ground truth*; tidak ada prediksi model.

**Cara pakai**
1. *Add Data*: `benyxu/dsf-yolo-steel-tube-dataset` dan `benyxu/swrd-dataset`. *Accelerator*: None (CPU cukup).
2. *Run All*. Sel 3 mencocokkan jumlah citra dan objek dengan angka naskah (berhenti bila tidak cocok) dan
   menentukan nama kelas SWRD dari jumlah objek per kelas di protokol tile v1.
3. Sel 6 menampilkan 8 kandidat per kelas dalam **urutan acak dengan seed tetap**. Kotak kuning = kelas yang
   sedang ditampilkan; kotak biru putus-putus = objek kelas lain; kotak putih bertitik = jendela perbesaran.
   Baris bawah = perbesaran pada kotak terbesar kelas itu. Keterangan: jumlah kotak kelas itu di citra, ukuran
   kotak dalam piksel masukan jaringan (sisi panjang citra = 640), rasio aspek, dan `tepi` bila kotak menyentuh
   batas citra/tile.
4. Bila belum ada yang cocok untuk suatu kelas, isi `HALAMAN_KHUSUS` di Sel 1 (mis. `{"SP-crack": 1}` = 8 kandidat
   berikutnya dari urutan acak yang sama), set `TAMPILKAN = ["SP-crack"]` agar hanya kelas itu yang digambar,
   lalu jalankan ulang Sel 1 dan Sel 6.
5. Isi `PILIHAN` di Sel 7 dengan ID kandidat (mis. `"SP-crack-003"`), satu per kelas; jalankan Sel 7 dan Sel 8.
6. Unduh `/kaggle/working/pilihan_contoh_dataset.zip` dan lampirkan ke Claude. Gambar final dibuat ulang
   dengan gaya naskah (font, ukuran kolom IEEE) dari berkas itu.

**Aturan pemilihan (untuk caption).** Kandidat suatu kelas = semua citra split validasi yang memuat kelas itu,
diurutkan acak dengan seed 2026; penulis memilih dari urutan itu. ID kandidat memuat peringkatnya dalam urutan
acak, sehingga pilihan dapat dilacak. Citra ditampilkan apa adanya seperti masuk ke jaringan (SWRD: setelah
*contrast stretching* dan CLAHE yang menjadi bagian pembuatan tile), tanpa peningkatan kontras tambahan.

In [ ]:
# === Sel 1. Konfigurasi ===
import os, glob, json, math, shutil, hashlib, csv
from collections import Counter, defaultdict
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

INPUT_ROOT = os.environ.get("NB15_INPUT_ROOT", "/kaggle/input")
OUT_DIR    = os.environ.get("NB15_OUT_DIR", "/kaggle/working")

SPLIT_TAMPIL   = "val"   # "val" (disarankan; sama dgn split evaluasi) atau "train"
SEED           = 2026
N_PER_HALAMAN  = 8
HALAMAN        = 0       # halaman default utk semua kelas (0 = 8 kandidat pertama urutan acak)
HALAMAN_KHUSUS = {}      # mis. {"SP-crack": 1, "SW-undercut": 2}
TAMPILKAN      = None    # None = semua kelas; atau mis. ["SP-crack", "SW-porosity"]
IMGSZ          = 640     # sisi panjang masukan jaringan (Ultralytics: resize seragam, sisi panjang = 640)
ZOOM_FAKTOR    = 3.0     # sisi jendela perbesaran = 3 x sisi panjang kotak ...
ZOOM_MIN_IN    = 64      # ... minimal 64 px masukan jaringan

STEEL_ROOT = None        # isi manual bila deteksi otomatis di Sel 2 gagal (folder yang berisi images/ & labels/)
SWRD_ROOT  = None
os.makedirs(OUT_DIR, exist_ok=True)
print("INPUT_ROOT =", INPUT_ROOT, "| OUT_DIR =", OUT_DIR, "| split =", SPLIT_TAMPIL, "| seed =", SEED)

In [ ]:
# === Sel 2. Temukan folder dataset (format YOLO) di bawah INPUT_ROOT ===
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def _punya_citra(d):
    try:
        with os.scandir(d) as it:
            return any(e.is_file() and os.path.splitext(e.name)[1].lower() in IMG_EXT for e in it)
    except OSError:
        return False

def _jumlah_citra(d):
    with os.scandir(d) as it:
        return sum(1 for e in it if e.is_file() and os.path.splitext(e.name)[1].lower() in IMG_EXT)

def temukan_split_yolo(root):
    # Tata letak A: <root>/images/<split> + <root>/labels/<split>;  B: <root>/<split>/images + <root>/<split>/labels
    hasil = []
    for dp, dns, _ in os.walk(root):
        if os.path.basename(dp).lower() != "images":
            continue
        parent = os.path.dirname(dp)
        lab = os.path.join(parent, "labels")
        for s in dns:
            if os.path.isdir(os.path.join(lab, s)) and _punya_citra(os.path.join(dp, s)):
                hasil.append((parent, s, os.path.join(dp, s), os.path.join(lab, s)))
        if os.path.isdir(lab) and _punya_citra(dp):
            hasil.append((os.path.dirname(parent), os.path.basename(parent), dp, lab))
    return hasil

GRUP = defaultdict(dict)
for r, s, di, dl in temukan_split_yolo(INPUT_ROOT):
    GRUP[r][s] = (di, dl)
assert GRUP, f"Tidak ada folder images/ + labels/ di bawah {INPUT_ROOT}. Sudah Add Data kedua dataset?"
for r in sorted(GRUP):
    print(r)
    for s, (di, _) in sorted(GRUP[r].items()):
        print(f"    split {s!r}: {_jumlah_citra(di)} citra")

def _split(r, kata):
    c = sorted(s for s in GRUP[r] if kata in s.lower())
    return c[0] if len(c) == 1 else None

if STEEL_ROOT is None:
    _c = [r for r in GRUP if "train2021" in GRUP[r] and "val2021" in GRUP[r]]
    _c.sort(key=lambda r: ("/yolo/" not in r + "/", r.count("/"), r))
    assert _c, "Dataset steel-pipe (split train2021/val2021) tidak ditemukan; isi STEEL_ROOT di Sel 1."
    STEEL_ROOT = _c[0]
STEEL_SPLIT = {"train": "train2021", "val": "val2021"}

if SWRD_ROOT is None:
    _c = [r for r in GRUP if r != STEEL_ROOT and _split(r, "train") and _split(r, "val")
          and _jumlah_citra(GRUP[r][_split(r, "val")][0]) == 700]
    assert len(_c) >= 1, "Dataset SWRD tile v1 (700 tile validasi) tidak ditemukan; isi SWRD_ROOT di Sel 1."
    if len(_c) > 1:
        print("  (beberapa kandidat root SWRD; dipilih yang pertama):", _c)
    SWRD_ROOT = sorted(_c)[0]
SWRD_SPLIT = {"train": _split(SWRD_ROOT, "train"), "val": _split(SWRD_ROOT, "val")}
print("\nSTEEL_ROOT =", STEEL_ROOT, STEEL_SPLIT)
print("SWRD_ROOT  =", SWRD_ROOT, SWRD_SPLIT)

In [ ]:
# === Sel 3. Muat label + TRIPWIRE terhadap angka naskah; nama kelas SWRD dari jumlah objek ===
def muat_split(dir_img, dir_lab):
    recs = []
    for e in sorted(os.scandir(dir_img), key=lambda e: e.name):
        if not e.is_file() or os.path.splitext(e.name)[1].lower() not in IMG_EXT:
            continue
        stem = os.path.splitext(e.name)[0]
        lp = os.path.join(dir_lab, stem + ".txt")
        boxes = []
        if os.path.exists(lp):
            for ln in open(lp).read().splitlines():
                r = ln.split()
                if not r:
                    continue
                assert len(r) == 5, f"Baris label bukan kotak YOLO (5 kolom): {lp}: {ln!r}"
                boxes.append((int(float(r[0])), *map(float, r[1:])))
        with Image.open(e.path) as im:
            W, H = im.size
        recs.append(dict(path=e.path, file=e.name, stem=stem, W=W, H=H, lab=lp if os.path.exists(lp) else None,
                         boxes=boxes))
    return recs

def n_obj(recs):
    return sum(len(r["boxes"]) for r in recs)

# ---- steel-pipe (Yang et al. 2021, rilis apa adanya) ----
# Nama indeks -> nama asli penulis (diverifikasi Sesi 3 dari pasangan VOC-XML <-> label YOLO).
# CSV/YAML lama proyek memakai nama lama utk indeks 4, 5, 7 (dense-gas-hole, non-fusion, unfused-hole).
STEEL_NAMES = ["air-hole", "bite-edge", "broken-arc", "crack", "hollow-bead", "overlap", "slag-inclusion", "unfused"]
steel = {k: muat_split(*GRUP[STEEL_ROOT][s]) for k, s in STEEL_SPLIT.items()}
_ni = {k: len(v) for k, v in steel.items()}
_no = {k: n_obj(v) for k, v in steel.items()}
assert _ni == {"train": 2726, "val": 682}, f"steel-pipe: jumlah citra {_ni}, naskah 2,726 + 682"
assert _no["train"] + _no["val"] == 6783 and _no["val"] == 1381, f"steel-pipe: objek {_no}, naskah 6,783 (1,381 val)"
_pi = Counter(b[0] for v in steel.values() for r in v for b in r["boxes"])
assert set(_pi) <= set(range(8)), f"indeks kelas steel-pipe di luar 0..7: {sorted(_pi)}"
print("TRIPWIRE steel-pipe: COCOK (2,726 + 682 citra; 6,783 objek, 1,381 di val)")
README_COUNTS = [5191, 35, 458, 119, 229, 223, 120, 408]   # jumlah per kelas di README rilis
print("  objek per kelas (train+val):", {STEEL_NAMES[i]: _pi[i] for i in range(8)})
if [_pi[i] for i in range(8)] != README_COUNTS:
    print("  PERINGATAN: urutan jumlah per kelas berbeda dari README rilis. Nama kelas tetap dari pemetaan VOC-XML,"
          " tetapi laporkan ke Claude sebelum memakai gambar.")

# ---- SWRD tile v1 (builder v4.1) ----
SWRD_HARAPAN = {   # (objek train, objek val) dari claude/swrd-tiles-v1-protokol.md; nama = nama di naskah
    "porosity": (2204, 619), "inclusion": (254, 78), "crack": (244, 85),
    "undercut": (124, 14), "lack of fusion": (130, 47), "lack of penetration": (198, 41)}
swrd = {k: muat_split(*GRUP[SWRD_ROOT][s]) for k, s in SWRD_SPLIT.items()}
_ni = {k: len(v) for k, v in swrd.items()}
_np_ = {k: sum(1 for r in v if r["boxes"]) for k, v in swrd.items()}
_no = {k: n_obj(v) for k, v in swrd.items()}
assert _ni == {"train": 2700, "val": 700}, f"SWRD: jumlah tile {_ni}, naskah 2,700 + 700"
assert _np_ == {"train": 1350, "val": 350}, f"SWRD: tile ber-cacat {_np_}, naskah 1,350 + 350"
assert _no == {"train": 3154, "val": 884}, f"SWRD: objek {_no}, naskah 3,154 + 884"
_cnt = {k: Counter(b[0] for r in v for b in r["boxes"]) for k, v in swrd.items()}
_ids = sorted(set(_cnt["train"]) | set(_cnt["val"]))
assert len(_ids) == 6, f"SWRD: {len(_ids)} id kelas, harus 6"
SWRD_NAMES = {}
for i in _ids:
    tup = (_cnt["train"][i], _cnt["val"][i])
    m = [n for n, t in SWRD_HARAPAN.items() if t == tup]
    assert len(m) == 1, f"SWRD id {i}: objek (train, val) = {tup} tidak cocok dgn protokol tile v1"
    SWRD_NAMES[i] = m[0]
assert sorted(SWRD_NAMES.values()) == sorted(SWRD_HARAPAN), "pemetaan nama SWRD tidak satu-satu"
print("TRIPWIRE SWRD: COCOK (2,700 + 700 tile; 1,350 + 350 ber-cacat; 3,154 + 884 objek)")
print("  id kelas -> nama (dari jumlah objek per kelas):", SWRD_NAMES)

# Cek silang dgn berkas nama kelas di dataset, bila ada (informasi saja)
def _norm(s):
    return str(s).lower().replace("_", " ").replace("-", " ").strip()
_nf = [p for pat in ("*.yaml", "*.yml", "classes.txt") for d in (SWRD_ROOT, os.path.dirname(SWRD_ROOT))
       for p in glob.glob(os.path.join(d, pat))]
for p in _nf:
    try:
        if p.endswith("classes.txt"):
            nm = [l.strip() for l in open(p) if l.strip()]
        else:
            import yaml
            nm = yaml.safe_load(open(p)).get("names")
        if isinstance(nm, dict):
            nm = [nm[k] for k in sorted(nm)]
        if not nm:
            continue
        ok = all(_norm(nm[i]) == _norm(SWRD_NAMES[i]) for i in SWRD_NAMES if i < len(nm))
        print(f"  {os.path.basename(p)}: {nm} -> {'SESUAI' if ok else 'PERINGATAN: BERBEDA dari pemetaan jumlah objek'}")
    except Exception as ex:
        print(f"  (tidak dapat membaca {p}: {ex})")

In [ ]:
# === Sel 4. Geometri kotak (piksel masukan jaringan) + cek silang dgn naskah (informasi) ===
def geom(rec, b):
    c, xc, yc, w, h = b
    W, H = rec["W"], rec["H"]
    s = IMGSZ / max(W, H)
    x1, x2, y1, y2 = (xc - w / 2) * W, (xc + w / 2) * W, (yc - h / 2) * H, (yc + h / 2) * H
    wi, hi = w * W * s, h * H * s
    return dict(c=c, x1=x1, y1=y1, x2=x2, y2=y2, w_in=wi, h_in=hi, luas=wi * hi,
                ar=max(wi, hi) / max(min(wi, hi), 1e-9), sisi=math.sqrt(wi * hi),
                tepi=(x1 <= 0.5 or y1 <= 0.5 or x2 >= W - 0.5 or y2 >= H - 0.5))

def _ringkas(nama, data, harap):
    g = [geom(r, b) for v in data.values() for r in v for b in r["boxes"]]
    ar = np.array([x["ar"] for x in g]); sisi = np.array([x["sisi"] for x in g])
    nilai = {"AR>5": f"{(ar > 5).sum()} ({100 * (ar > 5).mean():.2f}%)", "median sisi ekuivalen":
             f"{np.median(sisi):.1f} px", "p90 AR": f"{np.quantile(ar, .9):.2f}"}
    print(f"{nama} (train+val, {len(g)} objek):")
    for k, v in nilai.items():
        print(f"    {k:22s} {v:16s} naskah: {harap[k]}")

_ringkas("steel-pipe", steel, {"AR>5": "17 (0.25%)", "median sisi ekuivalen": "10.0 px", "p90 AR": "1.47"})
_ringkas("SWRD", swrd, {"AR>5": "662 (16.4%)", "median sisi ekuivalen": "40.5 px", "p90 AR": "8.42"})
print("(Bila ada yang berbeda, beri tahu Claude; definisi ukuran di sini = piksel masukan jaringan.)")

print(f"\nPer kelas, split {SPLIT_TAMPIL!r}:")
_swrd_urut = {i: SWRD_NAMES[i] for i in sorted(SWRD_NAMES, key=lambda i: list(SWRD_HARAPAN).index(SWRD_NAMES[i]))}
for nama, data, names in [("steel-pipe", steel, dict(enumerate(STEEL_NAMES))), ("SWRD", swrd, _swrd_urut)]:
    print(f"  {nama}")
    for cid, kelas in names.items():
        rr = [r for r in data[SPLIT_TAMPIL] if any(b[0] == cid for b in r["boxes"])]
        g = [geom(r, b) for r in rr for b in r["boxes"] if b[0] == cid]
        print(f"    {kelas:20s} {len(rr):4d} citra, {len(g):4d} kotak | median AR {np.median([x['ar'] for x in g]):5.2f}"
              f" | median sisi {np.median([x['sisi'] for x in g]):6.1f} px | menyentuh tepi "
              f"{100 * np.mean([x['tepi'] for x in g]):5.1f}%")

In [ ]:
# === Sel 5. Urutan acak kandidat per kelas (seed tetap) ===
DATASET = {
    "SP": dict(nama="steel-pipe", rec=steel[SPLIT_TAMPIL], names=dict(enumerate(STEEL_NAMES)), root=STEEL_ROOT),
    "SW": dict(nama="SWRD", rec=swrd[SPLIT_TAMPIL], names=_swrd_urut, root=SWRD_ROOT),
}
def slug(n):
    return n.replace(" ", "_")

KANDIDAT, URUTAN, KELAS_KUNCI = {}, {}, {}
for di, (ds, D) in enumerate(DATASET.items()):
    for cid, kelas in D["names"].items():
        pool = sorted((r for r in D["rec"] if any(b[0] == cid for b in r["boxes"])), key=lambda r: r["stem"])
        perm = np.random.default_rng([SEED, di, cid]).permutation(len(pool))
        ids = []
        for rank, j in enumerate(perm, 1):
            kid = f"{ds}-{slug(kelas)}-{rank:03d}"
            KANDIDAT[kid] = dict(ds=ds, cid=cid, kelas=kelas, rank=rank, pool=len(pool), rec=pool[j])
            ids.append(kid)
        URUTAN[f"{ds}-{slug(kelas)}"] = ids
        KELAS_KUNCI[f"{ds}-{slug(kelas)}"] = kelas
        print(f"  {ds}-{slug(kelas):22s} pool {len(pool):4d} citra -> {-(-len(pool) // N_PER_HALAMAN)} halaman")

with open(os.path.join(OUT_DIR, "kandidat_urutan_acak.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "dataset", "kelas", "rank_acak", "pool", "split", "seed", "file"])
    for kid, k in KANDIDAT.items():
        w.writerow([kid, DATASET[k["ds"]]["nama"], k["kelas"], k["rank"], k["pool"], SPLIT_TAMPIL, SEED, k["rec"]["file"]])
print("-> kandidat_urutan_acak.csv")

In [ ]:
# === Sel 6. Grid kandidat per kelas ===
WARNA_TARGET, WARNA_LAIN = "#ffd400", "#33c3ff"
_cache = {}

def muat_abu(path):
    if path not in _cache:
        im = Image.open(path)
        if im.mode in ("I;16", "I;16B", "I", "F"):   # jaga-jaga: citra 16-bit -> regangkan persentil 0.5-99.5
            a = np.asarray(im, dtype=np.float32)
            lo, hi = np.percentile(a, [0.5, 99.5])
            a = (np.clip((a - lo) / max(hi - lo, 1e-9), 0, 1) * 255).astype(np.uint8)
        else:
            a = np.asarray(im.convert("L"))
        if len(_cache) > 64:
            _cache.clear()
        _cache[path] = a
    return _cache[path]

def kotak_target(k, idx=0):
    g = sorted((geom(k["rec"], b) for b in k["rec"]["boxes"] if b[0] == k["cid"]), key=lambda x: -x["luas"])
    i = min(idx, len(g) - 1)
    return g[i], len(g), i

def jendela_zoom(rec, g):
    # Persegi berpusat di kotak, sisi = ZOOM_FAKTOR x sisi panjang kotak (min ZOOM_MIN_IN px masukan jaringan).
    # Bila persegi itu tidak muat di citra (kotak besar/memanjang), jendela = seluruh citra. Hasil: (x0, y0, lebar, tinggi).
    s = IMGSZ / max(rec["W"], rec["H"])
    sisi = max(ZOOM_FAKTOR * max(g["w_in"], g["h_in"]), ZOOM_MIN_IN) / s
    if sisi >= min(rec["W"], rec["H"]):
        return 0.0, 0.0, float(rec["W"]), float(rec["H"])
    cx, cy = (g["x1"] + g["x2"]) / 2, (g["y1"] + g["y2"]) / 2
    x0 = min(max(cx - sisi / 2, 0), rec["W"] - sisi)
    y0 = min(max(cy - sisi / 2, 0), rec["H"] - sisi)
    return x0, y0, sisi, sisi

def gambar_citra(ax, k, lw=1.2, jendela=None):
    rec = k["rec"]
    ax.imshow(muat_abu(rec["path"]), cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    for b in rec["boxes"]:
        g = geom(rec, b)
        t = b[0] == k["cid"]
        r = mpatches.Rectangle((g["x1"], g["y1"]), g["x2"] - g["x1"], g["y2"] - g["y1"], fill=False,
                               edgecolor=WARNA_TARGET if t else WARNA_LAIN, lw=lw if t else 0.8 * lw,
                               ls="-" if t else (0, (3, 2)))
        r.set_path_effects([pe.withStroke(linewidth=lw + 1.4, foreground="black")])
        ax.add_patch(r)
    if jendela is not None and (jendela[2] < rec["W"] or jendela[3] < rec["H"]):
        x0, y0, jw, jh = jendela
        ax.add_patch(mpatches.Rectangle((x0, y0), jw, jh, fill=False, edgecolor="white", lw=0.9, ls=(0, (1, 1.5))))
    ax.set_xticks([]); ax.set_yticks([])

def grid_kelas(key, halaman):
    ids = URUTAN[key][halaman * N_PER_HALAMAN:(halaman + 1) * N_PER_HALAMAN]
    if not ids:
        print(f"{key}: halaman {halaman} kosong (pool {len(URUTAN[key])} citra)")
        return
    k0 = KANDIDAT[ids[0]]
    ncol, nbar = 4, -(-len(ids) // 4)
    fig, ax = plt.subplots(2 * nbar, ncol, figsize=(4.3 * ncol, 3.1 * 2 * nbar), squeeze=False)
    for a in ax.ravel():
        a.axis("off")
    for n, kid in enumerate(ids):
        k = KANDIDAT[kid]
        g, nt, _ = kotak_target(k)
        jw = jendela_zoom(k["rec"], g)
        r, c = 2 * (n // ncol), n % ncol
        a1, a2 = ax[r, c], ax[r + 1, c]
        a1.axis("on"); a2.axis("on")
        gambar_citra(a1, k, jendela=jw)
        a1.set_title(f"{kid}\n{k['rec']['file'][:40]}", fontsize=8)
        gambar_citra(a2, k, lw=1.6)
        x0, y0, w_, h_ = jw
        a2.set_xlim(x0, x0 + w_); a2.set_ylim(y0 + h_, y0)
        a2.set_xlabel(f"{nt} kotak | {g['w_in']:.0f}x{g['h_in']:.0f} px@640 | AR {g['ar']:.1f}"
                      + (" | tepi" if g["tepi"] else ""), fontsize=8)
    D = DATASET[k0["ds"]]
    fig.suptitle(f"{D['nama']} | {k0['kelas']} | halaman {halaman} | pool {k0['pool']} citra ({SPLIT_TAMPIL}), "
                 f"urutan acak seed {SEED}", fontsize=11)
    fig.tight_layout()
    os.makedirs(os.path.join(OUT_DIR, "kandidat"), exist_ok=True)
    fig.savefig(os.path.join(OUT_DIR, "kandidat", f"{key}_h{halaman}.png"), dpi=110)
    plt.show()
    plt.close(fig)

for key in URUTAN:
    if TAMPILKAN is None or key in TAMPILKAN:
        grid_kelas(key, HALAMAN_KHUSUS.get(key, HALAMAN))

## Sel 7. Pilihan
Isi `PILIHAN` dengan satu ID kandidat per kelas (ID tertera di judul tiap kandidat). Kelas yang belum dipilih
tampil kosong di pratinjau. `ZOOM_KOTAK` opsional: bila citra memuat beberapa kotak kelas itu, `{"SP-crack-003": 1}`
memakai kotak terbesar ke-2 (0 = terbesar) untuk perbesaran.

In [ ]:
# === Sel 7. Pilihan penulis ===
PILIHAN = [
    # "SP-air-hole-001", "SP-bite-edge-001", "SP-broken-arc-001", "SP-crack-001",
    # "SP-hollow-bead-001", "SP-overlap-001", "SP-slag-inclusion-001", "SP-unfused-001",
    # "SW-porosity-001", "SW-inclusion-001", "SW-crack-001", "SW-undercut-001",
    # "SW-lack_of_fusion-001", "SW-lack_of_penetration-001",
]
ZOOM_KOTAK = {}

_salah = [p for p in PILIHAN if p not in KANDIDAT]
assert not _salah, f"ID tidak dikenal: {_salah} (cek ejaan; contoh format: SP-crack-003)"
_per = Counter(p.rsplit("-", 1)[0] for p in PILIHAN)
_ganda = [k for k, n in _per.items() if n > 1]
assert not _ganda, f"lebih dari satu pilihan untuk: {_ganda}"
_belum = [k for k in URUTAN if k not in _per]
print(f"{len(PILIHAN)} pilihan; belum dipilih: {_belum if _belum else 'tidak ada'}")

In [ ]:
# === Sel 8. Pratinjau susunan gambar + ekspor (zip) ===
from datetime import datetime, timezone
EXP = os.path.join(OUT_DIR, "pilihan_contoh_dataset")
if os.path.exists(EXP):
    shutil.rmtree(EXP)
os.makedirs(EXP)
pilih = {p.rsplit("-", 1)[0]: p for p in PILIHAN}

def panel(ax, key, zoom):
    ax.set_xticks([]); ax.set_yticks([])
    kelas = KELAS_KUNCI[key]
    if key not in pilih:
        ax.text(0.5, 0.5, "(belum\ndipilih)", ha="center", va="center", fontsize=7, transform=ax.transAxes)
        ax.set_box_aspect(1)
        ax.set_xlabel(kelas, fontsize=7)
        return None
    k = KANDIDAT[pilih[key]]
    g, _, i = kotak_target(k, ZOOM_KOTAK.get(pilih[key], 0))
    jw = jendela_zoom(k["rec"], g)
    gambar_citra(ax, k, lw=0.9)
    if zoom:
        x0, y0, w_, h_ = jw
        ax.set_xlim(x0, x0 + w_); ax.set_ylim(y0 + h_, y0)
    ax.set_xlabel(f"{kelas}\nAR {g['ar']:.1f}", fontsize=7, linespacing=1.1)
    return dict(k=k, g=g, idx=i, jendela=jw)

kunci_sp = [k for k in URUTAN if k.startswith("SP-")]
kunci_sw = [k for k in URUTAN if k.startswith("SW-")]
fig = plt.figure(figsize=(7.16, 3.6))
sub = fig.subfigures(2, 1, height_ratios=[1, 1.25])
info = {}
for sf, kunci, zoom, judul in [(sub[0], kunci_sp, True, "(a) Steel-pipe weld dataset (crops around the box)"),
                               (sub[1], kunci_sw, False, "(b) SWRD tiles (full tile)")]:
    axs = sf.subplots(1, len(kunci))
    for a, key in zip(np.atleast_1d(axs), kunci):
        info[key] = panel(a, key, zoom)
    sf.suptitle(judul, fontsize=8, x=0.01, ha="left")
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(EXP, f"pratinjau.{ext}"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

meta = dict(dibuat=datetime.now(timezone.utc).isoformat(timespec="seconds"), split=SPLIT_TAMPIL, seed=SEED,
            imgsz=IMGSZ, zoom_faktor=ZOOM_FAKTOR, zoom_min_in=ZOOM_MIN_IN, steel_root=STEEL_ROOT, swrd_root=SWRD_ROOT,
            steel_names=STEEL_NAMES, swrd_names={str(k): v for k, v in SWRD_NAMES.items()},
            aturan=("kandidat = semua citra split ini yang memuat kelas tsb., diurutkan acak dgn seed; "
                    "penulis memilih dari urutan itu"), pilihan=[])
for key, v in info.items():
    if v is None:
        continue
    k, rec = v["k"], v["k"]["rec"]
    d = os.path.join(EXP, k["ds"])
    os.makedirs(d, exist_ok=True)
    shutil.copy2(rec["path"], os.path.join(d, rec["file"]))
    if rec["lab"]:
        shutil.copy2(rec["lab"], os.path.join(d, rec["stem"] + ".txt"))
    names = DATASET[k["ds"]]["names"]
    meta["pilihan"].append(dict(
        id=pilih[key], dataset=DATASET[k["ds"]]["nama"], kelas=k["kelas"], class_id=k["cid"], rank_acak=k["rank"],
        pool=k["pool"], file=rec["file"], sha256=sha256(rec["path"]), W=rec["W"], H=rec["H"],
        kotak_zoom=dict(idx=v["idx"], **{q: round(float(v["g"][q]), 3) for q in
                        ("x1", "y1", "x2", "y2", "w_in", "h_in", "ar", "sisi")}, tepi=bool(v["g"]["tepi"])),
        jendela_zoom_x0_y0_w_h=[round(float(q), 2) for q in v["jendela"]],
        semua_kotak=[dict(kelas=names[b[0]], **{q: round(float(geom(rec, b)[q]), 2) for q in ("x1", "y1", "x2", "y2")})
                     for b in rec["boxes"]]))
json.dump(meta, open(os.path.join(EXP, "pilihan.json"), "w"), indent=1, ensure_ascii=False)
shutil.copy2(os.path.join(OUT_DIR, "kandidat_urutan_acak.csv"), EXP)
z = shutil.make_archive(EXP, "zip", EXP)
print(f"{len(meta['pilihan'])} pilihan diekspor -> {z}")
print("Unduh berkas zip itu (panel Output) dan lampirkan ke Claude.")